In [1]:
import os
import re
import json
import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
)

from datasets import Dataset as HFDataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
df_primary = pd.read_csv("/kaggle/input/datasets/carameyyow/primary/data_primer_filtered.csv")

In [3]:
print(df_primary.columns.tolist())

['Platform', 'Rating', 'Review', 'Review_Clean', 'shadow_sentiment', 'confidence_sentiment', 'shadow_emotion', 'confidence_emotion']


In [4]:
# Hapus hasil shadow labeling lama
df_primary = df_primary.drop(
    columns=[
        "shadow_sentiment",
        "confidence_sentiment",
        "shadow_emotion",
        "confidence_emotion"
    ],
    errors="ignore"
)

# Rename supaya konsisten
df_primary = df_primary.rename(columns={
    "Review": "Customer Review"
})

# Pakai Review_Clean kalau sudah ada
df_primary["text_clean"] = (
    df_primary["Review_Clean"]
    .fillna("")
    .astype(str)
)

print(df_primary.head())

    Platform  Rating                                    Customer Review  \
0  Tokopedia       5  Toko langganan Kantor. Harga tdk terlalu mahal...   
1  Tokopedia       5  Seller fast respon, kualitasnya bagus juga unt...   
2  Tokopedia       3  Maaf nih yah  Udh berusaha aku buat nekan kaca...   
3  Tokopedia       5  barang sesuai pesanan dan pengiriman cepat. se...   
4  Tokopedia       5  sumpah ya, aku pake ini bagus banget omgg🩷🩷, k...   

                                        Review_Clean  \
0  Toko langganan Kantor. Harga tdk terlalu mahal...   
1  Seller fast respon, kualitasnya bagus juga unt...   
2  Maaf nih yah Udh berusaha aku buat nekan kaca ...   
3  barang sesuai pesanan dan pengiriman cepat. se...   
4  sumpah ya, aku pakai ini bagus banget omgg pin...   

                                          text_clean  
0  Toko langganan Kantor. Harga tdk terlalu mahal...  
1  Seller fast respon, kualitasnya bagus juga unt...  
2  Maaf nih yah Udh berusaha aku buat nekan kac

In [5]:
print(df_primary["text_clean"].dtype)
print(df_primary["text_clean"].isna().sum())
print(df_primary["text_clean"].head())

object
0
0    Toko langganan Kantor. Harga tdk terlalu mahal...
1    Seller fast respon, kualitasnya bagus juga unt...
2    Maaf nih yah Udh berusaha aku buat nekan kaca ...
3    barang sesuai pesanan dan pengiriman cepat. se...
4    sumpah ya, aku pakai ini bagus banget omgg pin...
Name: text_clean, dtype: object


# Labeling

In [6]:
with open("/kaggle/input/datasets/carameyyow/model-skenario-3/best_model_s3_emotion/best_model_info.json") as f:
    emotion_info = json.load(f)

with open("/kaggle/input/datasets/carameyyow/model-skenario-3/best_model_s3_sentiment/best_model_info.json") as f:
    sentiment_info = json.load(f)

emotion_labels = emotion_info["label_names"]
sentiment_labels = sentiment_info["label_names"]

print(emotion_labels)
print(sentiment_labels)

['Anger', 'Fear', 'Happy', 'Love', 'Sadness']
['Negative', 'Positive']


In [7]:
# load model
emotion_dir = "/kaggle/input/datasets/carameyyow/model-skenario-3/best_model_s3_emotion"
sentiment_dir = "/kaggle/input/datasets/carameyyow/model-skenario-3/best_model_s3_sentiment"

tokenizer_emotion = AutoTokenizer.from_pretrained(emotion_dir)
model_emotion = AutoModelForSequenceClassification.from_pretrained(emotion_dir).to(DEVICE)

tokenizer_sentiment = AutoTokenizer.from_pretrained(sentiment_dir)
model_sentiment = AutoModelForSequenceClassification.from_pretrained(sentiment_dir).to(DEVICE)

trainer_emotion = Trainer(model=model_emotion)
trainer_sentiment = Trainer(model=model_sentiment)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [8]:
def build_hf_dataset(texts, labels, tokenizer, max_len=128):
    enc = tokenizer(
        list(texts),
        truncation=True,
        padding="max_length",
        max_length=max_len,
        return_tensors="pt"
    )

    return HFDataset.from_dict({
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "labels": np.asarray(labels, dtype=np.int64),
    })

## Emotion Prediction

In [9]:
dummy_labels = np.zeros(len(df_primary), dtype=int)

emotion_ds = build_hf_dataset(
    df_primary["text_clean"].values,
    dummy_labels,
    tokenizer_emotion
)

pred = trainer_emotion.predict(emotion_ds)

logits = pred.predictions
if isinstance(logits, tuple):
    logits = logits[0]

prob = torch.softmax(torch.tensor(logits), dim=1).numpy()

pred_idx = np.argmax(prob, axis=1)

df_primary["Emotion"] = [emotion_labels[i] for i in pred_idx]
df_primary["confidence_emotion"] = prob.max(axis=1)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


## Sentiment Prediction

In [10]:
sentiment_ds = build_hf_dataset(
    df_primary["text_clean"].values,
    dummy_labels,
    tokenizer_sentiment
)

pred = trainer_sentiment.predict(sentiment_ds)

logits = pred.predictions
if isinstance(logits, tuple):
    logits = logits[0]

prob = torch.softmax(torch.tensor(logits), dim=1).numpy()

idx = np.argmax(prob, axis=1)

df_primary["Sentiment"] = [sentiment_labels[i] for i in idx]
df_primary["confidence_sentiment"] = prob.max(axis=1)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [11]:
# save
df_primary.to_csv(
    "/kaggle/working/data_primer_labeled_best_model.csv",
    index=False
)

display(df_primary.head())

,Platform,Rating,Customer Review,Review_Clean,text_clean,Emotion,confidence_emotion,Sentiment,confidence_sentiment
0,Tokopedia,5,Toko langganan Kantor. Harga tdk terlalu mahal...,Toko langganan Kantor. Harga tdk terlalu mahal...,Toko langganan Kantor. Harga tdk terlalu mahal...,Love,0.591660,Positive,0.999983
1,Tokopedia,5,"Seller fast respon, kualitasnya bagus juga unt...","Seller fast respon, kualitasnya bagus juga unt...","Seller fast respon, kualitasnya bagus juga unt...",Happy,0.971432,Positive,0.999987
2,Tokopedia,3,Maaf nih yah Udh berusaha aku buat nekan kaca...,Maaf nih yah Udh berusaha aku buat nekan kaca ...,Maaf nih yah Udh berusaha aku buat nekan kaca ...,Fear,0.467770,Negative,0.997694
3,Tokopedia,5,barang sesuai pesanan dan pengiriman cepat. se...,barang sesuai pesanan dan pengiriman cepat. se...,barang sesuai pesanan dan pengiriman cepat. se...,Happy,0.978749,Positive,0.999987
4,Tokopedia,5,"sumpah ya, aku pake ini bagus banget omgg🩷🩷, k...","sumpah ya, aku pakai ini bagus banget omgg pin...","sumpah ya, aku pakai ini bagus banget omgg pin...",Love,0.873920,Positive,0.999986


In [12]:
df_primary.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32721 entries, 0 to 32720
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Platform              32721 non-null  object 
 1   Rating                32721 non-null  int64  
 2   Customer Review       32721 non-null  object 
 3   Review_Clean          32720 non-null  object 
 4   text_clean            32721 non-null  object 
 5   Emotion               32721 non-null  object 
 6   confidence_emotion    32721 non-null  float32
 7   Sentiment             32721 non-null  object 
 8   confidence_sentiment  32721 non-null  float32
dtypes: float32(2), int64(1), object(6)
memory usage: 2.0+ MB
